# MetabTravLR quickstart

Train SpaceTravLR with **harreman metabolite transporter pairs** added as a modulator
group, then read the learned `beta_<export>@<import>` coefficients back out over labeled
gene sets to rank metabolites by effect. We analyze the coefficients directly — **no
perturbation**.

Edit the **Config** and **Gene sets** cells, then run top to bottom. Everything is written
under the dataset directory. On Savio, replace `fit(...)` with the `spawn_worker` cell.

In [10]:
import os, sys
import numpy as np
import pandas as pd
import scanpy as sc

# SpaceTravLR package (src/) + our metab_processing helpers
_here = os.path.dirname(os.path.abspath('.'))
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
sys.path.append(os.path.join(os.getcwd(), '..'))

from SpaceTravLR.spaceship import SpaceShip
from metab_processing.SpaceTravLR.metab_loader import load_metab_pairs
from metab_processing.SpaceTravLR import beta_analysis

In [ ]:
# from harreman_summary import select_tcell_metabolites
# select_tcell_metabolites(f'{dataset_dir}/easy_download')

Wrote 76 metabolites -> /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Primary_Dermal_Melanoma/easy_download/harreman_outputs/metabolite_selection.yaml


PosixPath('/global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Primary_Dermal_Melanoma/easy_download/harreman_outputs/metabolite_selection.yaml')

## Config — data dir / dataset selection

Layout assumed: `DATA_DIR / DATASET / {adata.h5ad, easy_download/harreman_outputs/...}`.
Results are written to `DATA_DIR / DATASET / spacetravlr_output`.

In [4]:
DATA_DIR = '/global/scratch/users/fosterangus/MetabTravLR/Data/Xenium'
DATASET  = 'Primary_Dermal_Melanoma'   # dataset folder under DATA_DIR

CELL_TYPE_SRC = 'leiden_scVI_res_0.5'   # adata.obs column to use as 'cell_type' (the harreman tier annotation)

dataset_dir    = f'{DATA_DIR}/{DATASET}'
adata_path     = f'{dataset_dir}/adata.h5ad'
harreman_dir   = f'{dataset_dir}/easy_download/harreman_outputs'
selection_yaml = f'{harreman_dir}/metabolite_selection.yaml'
outdir         = f'{dataset_dir}/spacetravlr_output'
betadata_dir   = f'{outdir}/betadata'

for p in (adata_path, selection_yaml):
    assert os.path.exists(p), f'missing: {p}'

## Gene sets

The target genes to train and the labels to score metabolites against. `focus_genes` (the
genes actually trained) is the union of all sets. **Edit these lists.** With exactly
`positive`/`negative` labels, the ranking uses `signed = positive − negative`.

In [5]:
GENE_SETS = {
    'positive': ['CD4', 'CD3E', 'IL2RA'],          # e.g. T-cell activity
    'negative': ['CTLA4', 'FOXP3', 'IL10', 'ENTPD1'],  # e.g. exhaustion
}

focus_genes = list(dict.fromkeys(g for genes in GENE_SETS.values() for g in genes))
print(f'{len(focus_genes)} focus genes:', focus_genes)

7 focus genes: ['CD4', 'CD3E', 'IL2RA', 'CTLA4', 'FOXP3', 'IL10', 'ENTPD1']


In [6]:
adata = sc.read_h5ad(adata_path)
adata.obs['cell_type'] = adata.obs[CELL_TYPE_SRC]
adata.layers['raw_count'] = adata.X
adata

AnnData object with n_obs × n_vars = 112551 × 5006
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', '_scvi_batch', '_scvi_labels', 'leiden_scVI_res_0.5', 'leiden_scVI_res_2.5', 'leiden_scVI_res_2', 'leiden_scVI_res_1.5', 'leiden_scVI_res_1', 'leiden_scVI_res_0.75', 'leiden_scVI_res_0.65', 'leiden_scVI_res_0.375', 'leiden_scVI_res_0.25', 'leiden_scVI_res_0.1', 'leiden_scVI_res_0.05', 'cd8', 'cd4', 't_cell', 'Tier1', 'Tier2', 'Tier3', 'Cytotoxic_CD8_score', 'Exhausted_CD8_score', 'Treg_score', 'sub_cluster_5_res_1', 'sub_cluster_5_res_0.75', 'sub_cluster_5_res_0.5', 'sub_cluster_5_res_0.37', 'sub_cluster_5_res_0.25', 'sub_cluster_5_res_0.15', 'sub_cluster_5_res_0.1', 'sub_cluster_5_res_0.05', 'sub_cluster_2_res_1', 'sub_cluster_2_res_0.75', 'sub_cluster_2_res_0

## Metabolite pairs from harreman

`metabolite_selection.yaml` → the deduped `metab_pairs` list (homotypic once, heterotypic
both orientations) filtered to genes in the panel. `selection` keeps the metabolite→pairs
grouping for the read-back.

In [11]:
metab_pairs, selection = load_metab_pairs(selection_yaml, var_names=adata.var_names)
print(f'{len(selection)} metabolites, {len(metab_pairs)} model pairs (both orientations, in-panel)')
metab_pairs[:8]

build_metab_pairs: dropped 0 of 144 metab_pairs (gene absent from var_names); kept 144
76 metabolites, 144 model pairs (both orientations, in-panel)


[('ABCB1', 'ABCB1'),
 ('SLC15A1', 'SLC15A1'),
 ('ABCA1', 'ABCA1'),
 ('ATP7A', 'ATP7A'),
 ('ATP7A', 'ATP7B'),
 ('ATP7B', 'ATP7A'),
 ('SLC16A4', 'SLC16A4'),
 ('SLC16A4', 'SLCO2B1')]

## Setup + train

COMMOT is skipped — harreman is our metabolite prior. Only `focus_genes` are trained.

In [12]:
spacetravlr = SpaceShip(
    name=DATASET.replace('/', '_'),
    outdir=outdir,
    genes=focus_genes,
)

In [13]:
spacetravlr.setup_(adata, overwrite=False, run_commot=False)
assert spacetravlr.is_everything_ok()

AssertionError: Launch script not found

In [14]:
# Local / single-process training. On Savio use the spawn_worker cell below instead.
spacetravlr.fit(metab_pairs=metab_pairs)

Fitting ENTPD1 with 1252 modulators
	83 Transcription Factors
	638 Ligand-Receptor Pairs
	387 TranscriptionFactor-Ligand Pairs
	0 Extra modulators
	144 Metabolite Pairs
0: 0.6581 | 0.6443
1: 0.9241 | 0.9091
2: 0.7012 | 0.7302
3: 0.9581 | 0.9423
4: 0.7945 | 0.7428
5: 0.9738 | 0.9645
6: 0.3118 | 0.3106
7: 0.9353 | 0.9154
8: 0.5697 | 0.5665
9: 0.9354 | 0.9287
10: 0.7432 | 0.7069
Deleted lock for CD3E after 3600 seconds
Fitting CD3E with 1055 modulators
	47 Transcription Factors
	640 Ligand-Receptor Pairs
	224 TranscriptionFactor-Ligand Pairs
	0 Extra modulators
	144 Metabolite Pairs
0: 0.5713 | 0.5446
1: 0.7856 | 0.7812
2: 0.7087 | 0.6882
3: 0.8170 | 0.7577
4: 0.6562 | 0.6228
5: 0.6864 | 0.6030
6: 0.2848 | 0.2846
7: 0.7183 | 0.6715
8: 0.6232 | 0.6219
9: 0.5927 | 0.5567
10: 0.6982 | 0.6222


In [ ]:
# --- Savio: run this cell (multiple times) to spawn parallel SLURM workers instead of fit() ---
# spacetravlr.focus_genes = focus_genes
# spacetravlr.spawn_worker(
#     account='fc_wagnerlabfca',
#     partition='savio4_gpu',
#     qos='a5k_gpu4_normal',
#     gres='gpu:A5000:1',
#     job_name='MetabTravLR',
#     cpus_per_task=4,
#     lifespan=0.5,
#     python_path='/global/home/users/fosterangus/.conda/envs/spacetravlr/bin/python',
#     metab_pairs=metab_pairs,   # if driving via a launch.py, pass metab_pairs to run_spacetravlr
# )

## Read the metabolite coefficients back out

Per-`(gene, pair, cell_type)` beta summary → roll up to metabolites (optionally weighting
each transporter pair by its harreman `C_np` communication score) → signed gene-set ranking.

In [ ]:
# Per-(gene, export, import, cell_type) mean/std of the learned metabolite beta.
pair_summary = beta_analysis.read_metab_beta_summary(
    betadata_dir,
    genes=focus_genes,
    obs=adata.obs,
    cell_type_col='cell_type',
)
pair_summary.head()

,gene,export,import,pair,cell_type,mean,std,n,frac_nonzero
0,CD4,ABCA1,ABCA1,ABCA1@ABCA1,0,1.284072e-06,1.073941e-06,37999,0.996289
1,CD4,ATP7A,ATP7A,ATP7A@ATP7A,0,0.000000e+00,0.000000e+00,37999,0.000000
2,CD4,SLC16A4,SLCO2B1,SLC16A4@SLCO2B1,0,5.939908e-07,4.521215e-07,37999,1.000000
3,CD4,SLCO2B1,SLC16A4,SLCO2B1@SLC16A4,0,-2.606595e-06,1.958648e-06,37999,1.000000
4,CD4,SLCO2B1,SLCO2B1,SLCO2B1@SLCO2B1,0,3.279093e-06,9.167121e-07,37999,1.000000
...,...,...,...,...,...,...,...,...,...
3416,ENTPD1,SLCO2B1,ABCC1,SLCO2B1@ABCC1,10,0.000000e+00,0.000000e+00,521,0.000000
3417,ENTPD1,SLC16A1,ABCC1,SLC16A1@ABCC1,10,0.000000e+00,0.000000e+00,521,0.000000
3418,ENTPD1,ABCC4,SLCO2B1,ABCC4@SLCO2B1,10,0.000000e+00,0.000000e+00,521,0.000000
3419,ENTPD1,SLCO2B1,ABCC4,SLCO2B1@ABCC4,10,0.000000e+00,0.000000e+00,521,0.000000


In [18]:
# Optional: weight transporter pairs by harreman C_np (real-communication score).
# Set weights=None for a plain mean across a metabolite's pairs.
try:
    weights = beta_analysis.gene_pair_cnp_weights(harreman_dir, agg='max')
except Exception as e:
    print(f'C_np weights unavailable ({e}); falling back to unweighted mean')
    weights = None

metab_summary = beta_analysis.aggregate_to_metabolite(pair_summary, selection, weights=weights)
metab_summary

,metabolite,gene,cell_type,score,n_pairs
0,25-Hydroxycholesterol,CD3E,0,0.000000,1
1,25-Hydroxycholesterol,CD3E,1,-0.000006,1
2,25-Hydroxycholesterol,CD3E,10,0.000000,1
3,25-Hydroxycholesterol,CD3E,2,0.000000,1
4,25-Hydroxycholesterol,CD3E,3,0.000000,1
...,...,...,...,...,...
2140,alpha-Tocopherol,ENTPD1,5,0.000000,1
2141,alpha-Tocopherol,ENTPD1,6,0.000000,1
2142,alpha-Tocopherol,ENTPD1,7,0.000000,1
2143,alpha-Tocopherol,ENTPD1,8,0.000000,1


In [20]:
# Signed metabolite ranking: mean over 'positive' genes − mean over 'negative' genes.
ranking = beta_analysis.gene_set_score(metab_summary, GENE_SETS)
ranking.head(50)

,metabolite,cell_type,positive,negative,signed
0,Citric acid,2,9.602650e-03,-3.668500e-08,9.602686e-03
1,Oxalic acid,2,9.602650e-03,-3.668500e-08,9.602686e-03
2,Estradiol,2,7.943486e-03,-3.203718e-08,7.943518e-03
3,Nicotinate,2,3.599962e-03,-2.808889e-08,3.599990e-03
4,Taurocholic acid,2,2.597603e-03,-2.760613e-08,2.597630e-03
5,Acetate,2,2.504485e-03,-9.166094e-08,2.504576e-03
6,Iron,2,1.588531e-03,-3.145593e-08,1.588562e-03
7,Adenosine triphosphate,2,1.348374e-03,-9.184541e-08,1.348466e-03
8,Guanosine,2,1.019561e-03,2.166818e-08,1.019539e-03
9,Cytidine,2,1.019561e-03,2.166818e-08,1.019539e-03
